# Yelp Final Content-Based Recommendation System for Amazon EMR

## Purpose

This notebook builds the complete content-based recommendation component for the Yelp RAG chatbot.

The production flow is:

1. Read business-content and user-review interaction data from Amazon S3.
2. Treat ratings of 4 or 5 as positive preference signals.
3. Build normalized business-content vectors.
4. Create separate validation and test users from held-out positive interactions.
5. Tune hyperparameters only on validation users using compact grid search.
6. Evaluate the selected model once on untouched test users.
7. Generate recommendations from several liked businesses, not only one anchor.
8. Exclude businesses the user has already rated.
9. Add controlled diversity and a popularity fallback so the result is not unnecessarily empty.
10. Save model artifacts and RAG-ready recommendation output to Amazon S3.

> The business feature pipeline is unsupervised and may be fitted on the complete business catalogue. User interactions are split because they provide the preference labels used for tuning and evaluation.


## Step 1 — Import PySpark libraries and configure the EMR Spark session

Adaptive Query Execution is enabled. Shuffle partitions should be adjusted according to the EMR cluster size.

In [ ]:
from functools import reduce
from math import log2

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql import types as T

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    RegexTokenizer,
    HashingTF,
    IDF,
    FeatureHasher,
    VectorAssembler,
    Normalizer,
    BucketedRandomProjectionLSH
)

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "400")
spark.conf.set("spark.serializer", "org.apache.spark.serializer.KryoSerializer")

print("Spark version:", spark.version)
print("EMR content-based recommendation notebook initialized.")

## Step 2 — Define S3 input and output paths

`BUSINESS_INPUT_PATH` is the business-content Parquet location already supplied.

`INTERACTIONS_INPUT_PATH` must contain at least:

- `user_id`
- `business_id`
- `stars`

The optional `date` column is used to break ties when selecting a user's most recent liked business.

In [ ]:
BUSINESS_INPUT_PATH = "s3://yelpdataset-project/gold_layer/ml/content_based_filtering/"
INTERACTIONS_INPUT_PATH = "s3://yelpdataset-project/gold_layer/ml/collaborative_filtering/"

OUTPUT_ROOT = "s3://yelpdataset-project/gold_layer/ml/content_based_model_final/"
FEATURES_OUTPUT_PATH = f"{OUTPUT_ROOT}business_features/"
PIPELINE_MODEL_PATH = f"{OUTPUT_ROOT}feature_pipeline/"
LSH_MODEL_PATH = f"{OUTPUT_ROOT}lsh_model/"
TUNING_RESULTS_PATH = f"{OUTPUT_ROOT}evaluation/hyperparameter_results/"
FINAL_METRICS_PATH = f"{OUTPUT_ROOT}evaluation/final_test_metrics/"
RAG_RECOMMENDATIONS_PATH = f"{OUTPUT_ROOT}rag/sample_user_recommendations/"
BEST_PARAMS_PATH = f"{OUTPUT_ROOT}evaluation/best_parameters/"

POSITIVE_RATING_THRESHOLD = 4.0
TOP_K = 10
VALIDATION_USER_LIMIT = 100
TEST_USER_LIMIT = 100
VALIDATION_PERCENT = 80
MAX_POSITIVE_ANCHORS = 5
SIMILAR_RESULT_RATIO = 0.70
RANDOM_SEED = 42
DISTANCE_THRESHOLD = 1.80

print("Business input:", BUSINESS_INPUT_PATH)
print("Interaction input:", INTERACTIONS_INPUT_PATH)
print("Output root:", OUTPUT_ROOT)


## Step 3 — Read the Gold-layer Parquet datasets

Only external I/O is wrapped in `try-except`, because S3 paths and IAM permissions are common failure points on EMR.

In [ ]:
try:
    business_raw_df = spark.read.parquet(BUSINESS_INPUT_PATH)
    interactions_raw_df = spark.read.parquet(INTERACTIONS_INPUT_PATH)
except Exception as error:
    print("Failed to read one or more S3 inputs.")
    print("Check the S3 paths and the EMR EC2 instance-profile permissions.")
    raise

print("Business rows:", business_raw_df.count())
print("Interaction rows:", interactions_raw_df.count())

business_raw_df.printSchema()
interactions_raw_df.printSchema()

## Step 4 — Validate required columns and Gold-layer data quality

The notebook stops if primary-key problems are found. It does not silently drop them, because such issues should be fixed upstream in Silver or Gold.

In [ ]:
BUSINESS_REQUIRED_COLUMNS = [
    "business_id", "categories", "city", "state", "latitude", "longitude",
    "stars", "review_count", "is_open",
    "attributes_restaurantspricerange2", "attributes_wifi",
    "attributes_outdoorseating"
]

INTERACTION_REQUIRED_COLUMNS = ["user_id", "business_id", "stars"]

missing_business_columns = [c for c in BUSINESS_REQUIRED_COLUMNS if c not in business_raw_df.columns]
missing_interaction_columns = [c for c in INTERACTION_REQUIRED_COLUMNS if c not in interactions_raw_df.columns]

if missing_business_columns:
    raise ValueError(f"Missing business columns: {missing_business_columns}")
if missing_interaction_columns:
    raise ValueError(f"Missing interaction columns: {missing_interaction_columns}")

null_or_blank_business_ids = business_raw_df.filter(
    F.col("business_id").isNull() | (F.trim(F.col("business_id")) == "")
).count()

duplicate_business_ids = (
    business_raw_df.groupBy("business_id").count().filter(F.col("count") > 1).count()
)

if null_or_blank_business_ids > 0:
    raise ValueError(f"Gold business data contains {null_or_blank_business_ids} null/blank business IDs.")
if duplicate_business_ids > 0:
    raise ValueError(f"Gold business data contains {duplicate_business_ids} duplicate business IDs.")

print("Gold-layer business validation passed.")

## Step 5 — Standardize business features without dropping businesses

Missing optional content is represented by `unknown`. Invalid numeric values are validated and replaced only where necessary so every valid business ID can receive a feature vector.

Latitude and longitude are retained for RAG output and future distance reranking. State is used as a strict candidate filter by default.

In [ ]:
def clean_attribute(column_name):
    value = F.lower(F.trim(F.coalesce(F.col(column_name).cast("string"), F.lit("unknown"))))
    value = F.regexp_replace(value, r"^u'", "")
    value = F.regexp_replace(value, r"^'|'$", "")
    return F.when(value.isin("none", "null", "nan", ""), F.lit("unknown")).otherwise(value)

business_clean_df = (
    business_raw_df
    .withColumn("categories_clean", F.lower(F.trim(F.coalesce(F.col("categories"), F.lit("unknown")))))
    .withColumn("city_clean", F.lower(F.trim(F.coalesce(F.col("city"), F.lit("unknown")))))
    .withColumn("state_clean", F.upper(F.trim(F.coalesce(F.col("state"), F.lit("UNKNOWN")))))
    .withColumn("price_range_clean", clean_attribute("attributes_restaurantspricerange2"))
    .withColumn("wifi_clean", clean_attribute("attributes_wifi"))
    .withColumn("outdoor_seating_clean", clean_attribute("attributes_outdoorseating"))
    .withColumn("is_open_clean", F.when(F.col("is_open").cast("int") == 1, F.lit("open")).otherwise(F.lit("closed")))
    .withColumn("stars_clean", F.when(F.col("stars").cast("double").between(1.0, 5.0), F.col("stars").cast("double")).otherwise(F.lit(0.0)))
    .withColumn("review_count_clean", F.when(F.col("review_count").cast("double") >= 0, F.col("review_count").cast("double")).otherwise(F.lit(0.0)))
    .withColumn("latitude_clean", F.col("latitude").cast("double"))
    .withColumn("longitude_clean", F.col("longitude").cast("double"))
    .withColumn("stars_scaled", F.col("stars_clean") / F.lit(5.0))
    .withColumn("review_count_log", F.log1p(F.col("review_count_clean")))
    .cache()
)

print("Prepared businesses:", business_clean_df.count())
business_clean_df.select(
    "business_id", "categories_clean", "city_clean", "state_clean",
    "price_range_clean", "wifi_clean", "outdoor_seating_clean",
    "stars_clean", "review_count_clean", "is_open_clean"
).show(10, truncate=False)

## Step 6 — Prepare user interactions

Ratings of 4 or 5 are positive preferences. All ratings are retained in `seen_interactions_df` so previously rated businesses can be excluded from recommendations.

In [ ]:
interaction_date_expr = (
    F.to_timestamp("date") if "date" in interactions_raw_df.columns
    else F.lit(None).cast("timestamp")
)

interactions_df = (
    interactions_raw_df
    .select(
        F.trim(F.col("user_id")).alias("user_id"),
        F.trim(F.col("business_id")).alias("business_id"),
        F.col("stars").cast("double").alias("user_rating"),
        interaction_date_expr.alias("interaction_date")
    )
    .filter(
        F.col("user_id").isNotNull() & (F.col("user_id") != "") &
        F.col("business_id").isNotNull() & (F.col("business_id") != "") &
        F.col("user_rating").between(1.0, 5.0)
    )
    .groupBy("user_id", "business_id")
    .agg(
        F.max("user_rating").alias("user_rating"),
        F.max("interaction_date").alias("interaction_date")
    )
    .join(business_clean_df.select("business_id"), "business_id", "inner")
    .cache()
)

seen_interactions_df = interactions_df.select("user_id", "business_id").distinct().cache()
positive_interactions_df = interactions_df.filter(
    F.col("user_rating") >= POSITIVE_RATING_THRESHOLD
).cache()

print("Valid unique user-business interactions:", interactions_df.count())
print("Positive interactions:", positive_interactions_df.count())

## Step 7 — Create separate validation and test cases

A proper recommendation evaluation must keep the final test users separate from hyperparameter selection.

For every eligible user with at least two positively rated businesses:

- one liked business is used as the anchor;
- another liked business is hidden as the relevant target.

Eligible users are deterministically divided into:

- **validation users:** used for hyperparameter grid search;
- **test users:** untouched during tuning and used once for final evaluation.

This is a user-level holdout, so the same user cannot appear in both sets.


In [ ]:
eligible_users_df = (
    positive_interactions_df.groupBy("user_id")
    .count()
    .filter(F.col("count") >= 2)
    .select("user_id")
)

rank_window = Window.partitionBy("user_id").orderBy(F.rand(RANDOM_SEED))

ranked_positive_df = (
    positive_interactions_df
    .join(eligible_users_df, "user_id", "inner")
    .withColumn("preference_rank", F.row_number().over(rank_window))
)

anchor_df = ranked_positive_df.filter(F.col("preference_rank") == 1).select(
    "user_id",
    F.col("business_id").alias("anchor_business_id")
)

target_df = ranked_positive_df.filter(F.col("preference_rank") == 2).select(
    "user_id",
    F.col("business_id").alias("target_business_id")
)

all_evaluation_cases_df = (
    anchor_df.join(target_df, "user_id", "inner")
    .join(
        business_clean_df.select(
            F.col("business_id").alias("anchor_business_id"),
            F.col("state_clean").alias("anchor_state")
        ),
        "anchor_business_id",
        "inner"
    )
    .join(
        business_clean_df.select(
            F.col("business_id").alias("target_business_id"),
            F.col("state_clean").alias("target_state"),
            F.col("is_open_clean").alias("target_open_status")
        ),
        "target_business_id",
        "inner"
    )
    .filter(
        (F.col("anchor_state") == F.col("target_state")) &
        (F.col("target_open_status") == "open")
    )
    .withColumn(
        "user_bucket",
        F.pmod(F.xxhash64("user_id"), F.lit(100))
    )
    .cache()
)

validation_cases_df = (
    all_evaluation_cases_df
    .filter(F.col("user_bucket") < VALIDATION_PERCENT)
    .orderBy(F.rand(RANDOM_SEED))
    .limit(VALIDATION_USER_LIMIT)
    .drop("user_bucket")
    .cache()
)

test_cases_df = (
    all_evaluation_cases_df
    .filter(F.col("user_bucket") >= VALIDATION_PERCENT)
    .orderBy(F.rand(RANDOM_SEED + 1))
    .limit(TEST_USER_LIMIT)
    .drop("user_bucket")
    .cache()
)

VALIDATION_CASE_COUNT = validation_cases_df.count()
TEST_CASE_COUNT = test_cases_df.count()

if VALIDATION_CASE_COUNT == 0:
    raise ValueError("No validation cases were created.")
if TEST_CASE_COUNT == 0:
    raise ValueError("No independent test cases were created.")

print("Validation users:", VALIDATION_CASE_COUNT)
print("Independent test users:", TEST_CASE_COUNT)


## Step 8 — Define the tunable feature pipeline

### FeatureHasher

`FeatureHasher` converts categorical metadata such as city, state, price range, Wi-Fi and outdoor seating into a fixed-size sparse vector. It is memory-efficient on Spark because it avoids creating one separate one-hot column for every possible value.

Its main limitation is hash collision: two values can map to the same vector position. The metadata vector size is therefore included in tuning.

### Normalizer with `p=2.0`

`p=2.0` means L2 or Euclidean normalization.

For a vector `[x1, x2, ..., xn]`, its length is:

`sqrt(x1² + x2² + ... + xn²)`

Every value is divided by that length, producing a unit-length vector. This prevents businesses with more categories or larger numeric values from automatically dominating the distance calculation.


In [ ]:
def build_feature_pipeline(category_num_features, metadata_num_features, min_doc_freq):
    category_tokenizer = RegexTokenizer(
        inputCol="categories_clean",
        outputCol="category_tokens",
        pattern=r",\s*",
        gaps=True,
        toLowercase=True,
        minTokenLength=1
    )

    category_tf = HashingTF(
        inputCol="category_tokens",
        outputCol="category_tf",
        numFeatures=int(category_num_features),
        binary=True
    )

    category_idf = IDF(
        inputCol="category_tf",
        outputCol="category_tfidf",
        minDocFreq=int(min_doc_freq)
    )

    metadata_hasher = FeatureHasher(
        inputCols=[
            "city_clean", "state_clean", "price_range_clean",
            "wifi_clean", "outdoor_seating_clean", "is_open_clean"
        ],
        outputCol="metadata_features",
        numFeatures=int(metadata_num_features),
        categoricalCols=[
            "city_clean", "state_clean", "price_range_clean",
            "wifi_clean", "outdoor_seating_clean", "is_open_clean"
        ]
    )

    assembler = VectorAssembler(
        inputCols=["category_tfidf", "metadata_features", "stars_scaled", "review_count_log"],
        outputCol="raw_features",
        handleInvalid="keep"
    )

    normalizer = Normalizer(inputCol="raw_features", outputCol="features", p=2.0)

    return Pipeline(stages=[
        category_tokenizer, category_tf, category_idf,
        metadata_hasher, assembler, normalizer
    ])

## Step 9 — Define batch Top-K evaluation metrics

Metrics:

- **Hit Rate@K:** percentage of users whose held-out liked business appears in Top-K;
- **Recall@K:** with one held-out target per user, equal to Hit Rate@K;
- **MRR@K:** rewards a relevant business appearing near the top;
- **NDCG@K:** position-sensitive ranking quality.

In [ ]:
def evaluate_model(feature_df, lsh_model, evaluation_cases, seen_df, k=10, distance_threshold=1.8):
    candidate_df = (
        feature_df.filter(F.col("is_open_clean") == "open")
        .select(
            "business_id", "state_clean", "features", "categories",
            "city", "state", "stars_clean", "review_count_clean",
            "price_range_clean", "wifi_clean", "outdoor_seating_clean"
        )
    )

    anchor_vectors_df = (
        evaluation_cases
        .join(
            feature_df.select(
                F.col("business_id").alias("anchor_business_id"),
                F.col("features").alias("features"),
                F.col("state_clean").alias("anchor_state_from_features")
            ),
            "anchor_business_id",
            "inner"
        )
        .select("user_id", "anchor_business_id", "target_business_id", "anchor_state", "features")
    )

    pairs_df = (
        lsh_model.approxSimilarityJoin(
            anchor_vectors_df,
            candidate_df,
            float(distance_threshold),
            distCol="distance"
        )
        .select(
            F.col("datasetA.user_id").alias("user_id"),
            F.col("datasetA.anchor_business_id").alias("anchor_business_id"),
            F.col("datasetA.target_business_id").alias("target_business_id"),
            F.col("datasetB.business_id").alias("recommended_business_id"),
            F.col("datasetB.state_clean").alias("candidate_state"),
            F.col("datasetA.anchor_state").alias("anchor_state"),
            F.col("distance")
        )
        .filter(
            (F.col("recommended_business_id") != F.col("anchor_business_id")) &
            (F.col("candidate_state") == F.col("anchor_state"))
        )
    )

    # During offline evaluation, the held-out target must remain eligible.
    # Remove all other previously seen businesses, but not the target itself.
    seen_for_evaluation_df = (
        seen_df
        .join(
            evaluation_cases.select("user_id", "target_business_id"),
            "user_id",
            "inner"
        )
        .filter(F.col("business_id") != F.col("target_business_id"))
        .select(
            "user_id",
            F.col("business_id").alias("recommended_business_id")
        )
        .distinct()
    )

    unseen_pairs_df = pairs_df.join(
        seen_for_evaluation_df,
        ["user_id", "recommended_business_id"],
        "left_anti"
    )

    ranking_window = Window.partitionBy("user_id").orderBy(
        F.asc("distance"), F.asc("recommended_business_id")
    )

    topk_df = (
        unseen_pairs_df
        .withColumn("rank", F.row_number().over(ranking_window))
        .filter(F.col("rank") <= int(k))
        .cache()
    )

    per_user_df = (
        evaluation_cases.select("user_id", "target_business_id")
        .join(
            topk_df.select(
                "user_id", "recommended_business_id", "rank"
            ),
            "user_id",
            "left"
        )
        .withColumn(
            "is_hit",
            F.when(F.col("recommended_business_id") == F.col("target_business_id"), 1.0).otherwise(0.0)
        )
        .groupBy("user_id", "target_business_id")
        .agg(
            F.max("is_hit").alias("hit"),
            F.min(F.when(F.col("is_hit") == 1.0, F.col("rank"))).alias("hit_rank")
        )
        .withColumn("reciprocal_rank", F.when(F.col("hit") == 1.0, 1.0 / F.col("hit_rank")).otherwise(0.0))
        .withColumn("ndcg", F.when(F.col("hit") == 1.0, 1.0 / F.log2(F.col("hit_rank") + 1.0)).otherwise(0.0))
    )

    metrics = per_user_df.agg(
        F.avg("hit").alias("hit_rate_at_k"),
        F.avg("hit").alias("recall_at_k"),
        F.avg("reciprocal_rank").alias("mrr_at_k"),
        F.avg("ndcg").alias("ndcg_at_k")
    ).first().asDict()

    metrics["evaluated_users"] = evaluation_cases.count()
    metrics["recommendation_rows"] = topk_df.count()
    return metrics, topk_df

## Step 10 — Hyperparameter tuning with compact grid search

The notebook uses **grid search**, not Gaussian-process or Bayesian optimization.

Each listed parameter combination is fitted and evaluated on validation users. The best configuration is selected primarily by NDCG@K, followed by Hit Rate@K and MRR@K.

A compact manual grid is used because every Spark experiment rebuilds TF-IDF vectors, fits LSH and performs distributed recommendation evaluation. This is transparent, reproducible and practical for the current EMR project size.


In [ ]:
PARAMETER_GRID = [
    {
        "category_num_features": 2048,
        "metadata_num_features": 512,
        "min_doc_freq": 2,
        "bucket_length": 1.0,
        "num_hash_tables": 4
    },
    {
        "category_num_features": 4096,
        "metadata_num_features": 1024,
        "min_doc_freq": 2,
        "bucket_length": 1.25,
        "num_hash_tables": 5
    },
    {
        "category_num_features": 4096,
        "metadata_num_features": 1024,
        "min_doc_freq": 5,
        "bucket_length": 1.5,
        "num_hash_tables": 6
    },
    {
        "category_num_features": 8192,
        "metadata_num_features": 2048,
        "min_doc_freq": 2,
        "bucket_length": 1.25,
        "num_hash_tables": 8
    }
]

print("Hyperparameter combinations:", len(PARAMETER_GRID))

In [ ]:
tuning_results = []
best_result = None
best_pipeline_model = None
best_lsh_model = None
best_feature_df = None

for experiment_id, params in enumerate(PARAMETER_GRID, start=1):
    print(f"\nRunning experiment {experiment_id}/{len(PARAMETER_GRID)}")
    print(params)

    pipeline = build_feature_pipeline(
        params["category_num_features"],
        params["metadata_num_features"],
        params["min_doc_freq"]
    )

    pipeline_model = pipeline.fit(business_clean_df)

    feature_df = (
        pipeline_model.transform(business_clean_df)
        .select(
            "business_id", "categories", "city", "state",
            "latitude_clean", "longitude_clean", "stars_clean",
            "review_count_clean", "is_open_clean", "state_clean",
            "price_range_clean", "wifi_clean", "outdoor_seating_clean",
            "category_tokens", "features"
        )
        .cache()
    )
    feature_df.count()

    lsh = BucketedRandomProjectionLSH(
        inputCol="features",
        outputCol="hashes",
        bucketLength=float(params["bucket_length"]),
        numHashTables=int(params["num_hash_tables"]),
        seed=RANDOM_SEED
    )
    lsh_model = lsh.fit(feature_df)

    metrics, _ = evaluate_model(
        feature_df,
        lsh_model,
        validation_cases_df,
        seen_interactions_df,
        k=TOP_K,
        distance_threshold=DISTANCE_THRESHOLD
    )

    result = {
        "experiment_id": experiment_id,
        **params,
        **metrics
    }
    tuning_results.append(result)
    print(result)

    score_tuple = (
        result["ndcg_at_k"] or 0.0,
        result["hit_rate_at_k"] or 0.0,
        result["mrr_at_k"] or 0.0
    )
    best_score_tuple = None if best_result is None else (
        best_result["ndcg_at_k"] or 0.0,
        best_result["hit_rate_at_k"] or 0.0,
        best_result["mrr_at_k"] or 0.0
    )

    if best_result is None or score_tuple > best_score_tuple:
        if best_feature_df is not None:
            best_feature_df.unpersist()
        best_result = result
        best_pipeline_model = pipeline_model
        best_lsh_model = lsh_model
        best_feature_df = feature_df
    else:
        feature_df.unpersist()

print("\nBest hyperparameters selected using validation users:")
print(best_result)

## Step 11 — Display and save tuning results

In [ ]:
tuning_results_df = spark.createDataFrame(tuning_results).orderBy(
    F.desc("ndcg_at_k"),
    F.desc("hit_rate_at_k"),
    F.desc("mrr_at_k")
)

tuning_results_df.show(truncate=False)

best_params_df = spark.createDataFrame([best_result])
best_params_df.show(truncate=False)

## Step 12 — Final evaluation on untouched test users

The best hyperparameters were selected using validation users only.

This step evaluates that selected model once on independent test users. These test results are the metrics that should be reported in the project documentation.


In [ ]:
final_metrics, final_test_recommendations_df = evaluate_model(
    best_feature_df,
    best_lsh_model,
    test_cases_df,
    seen_interactions_df,
    k=TOP_K,
    distance_threshold=DISTANCE_THRESHOLD
)

final_metrics.update({
    "top_k": TOP_K,
    "positive_rating_threshold": POSITIVE_RATING_THRESHOLD,
    "distance_threshold": DISTANCE_THRESHOLD,
    "validation_users": VALIDATION_CASE_COUNT,
    "test_users": TEST_CASE_COUNT,
    "tuning_method": "manual_compact_grid_search"
})

final_metrics_df = spark.createDataFrame([final_metrics])
final_metrics_df.show(truncate=False)


## Step 13 — Multi-anchor, unseen and diversity-aware recommendations

A user may like several kinds of businesses. Using only the single highest-rated business can make recommendations too narrow.

The production function therefore:

1. uses up to `MAX_POSITIVE_ANCHORS` highly rated businesses;
2. gathers similar candidates for every anchor;
3. excludes every business the user already rated;
4. reserves most slots for strong content similarity;
5. uses the remaining slots for controlled exploration among open, unseen businesses in the required state;
6. falls back to well-rated popular unseen businesses in that state when LSH returns too few candidates.

A previously disliked business is not recommended again merely to fill the list.


In [ ]:
def recommend_for_user(
    user_id,
    number_of_recommendations=10,
    requested_state=None,
    open_only=True
):
    liked_anchors = (
        positive_interactions_df
        .filter(F.col("user_id") == user_id)
        .join(
            best_feature_df.select(
                "business_id", "categories", "state_clean",
                "features", "category_tokens"
            ),
            "business_id",
            "inner"
        )
        .orderBy(
            F.desc("user_rating"),
            F.desc_nulls_last("interaction_date"),
            F.desc("business_id")
        )
        .limit(MAX_POSITIVE_ANCHORS)
        .collect()
    )

    if not liked_anchors:
        raise ValueError(
            f"User {user_id} has no rating >= {POSITIVE_RATING_THRESHOLD}. "
            "Use the cold-start category/state function."
        )

    target_state = (
        requested_state.strip().upper()
        if requested_state
        else liked_anchors[0]["state_clean"]
    )

    candidate_df = best_feature_df.filter(F.col("state_clean") == target_state)
    if open_only:
        candidate_df = candidate_df.filter(F.col("is_open_clean") == "open")

    already_seen_df = (
        seen_interactions_df
        .filter(F.col("user_id") == user_id)
        .select(F.col("business_id").alias("seen_business_id"))
    )

    per_anchor_candidates = []
    overfetch_per_anchor = max(int(number_of_recommendations) * 15, 100)

    for anchor in liked_anchors:
        anchor_weight = max(
            0.1,
            (float(anchor["user_rating"]) - 3.0) / 2.0
        )

        nearest = (
            best_lsh_model.approxNearestNeighbors(
                candidate_df,
                anchor["features"],
                overfetch_per_anchor,
                distCol="distance"
            )
            .filter(F.col("business_id") != anchor["business_id"])
            .withColumn(
                "anchor_similarity",
                1.0 / (1.0 + F.col("distance"))
            )
            .withColumn(
                "weighted_similarity",
                F.col("anchor_similarity") * F.lit(anchor_weight)
            )
            .withColumn(
                "source_anchor_business_id",
                F.lit(anchor["business_id"])
            )
        )
        per_anchor_candidates.append(nearest)

    combined_candidates_df = reduce(
        lambda left, right: left.unionByName(right),
        per_anchor_candidates
    )

    unseen_content_df = (
        combined_candidates_df
        .join(
            already_seen_df,
            F.col("business_id") == F.col("seen_business_id"),
            "left_anti"
        )
        .groupBy(
            "business_id", "categories", "city", "state",
            "latitude_clean", "longitude_clean", "stars_clean",
            "review_count_clean", "price_range_clean",
            "wifi_clean", "outdoor_seating_clean",
            "is_open_clean", "state_clean", "category_tokens"
        )
        .agg(
            F.max("weighted_similarity").alias("content_score"),
            F.min("distance").alias("distance"),
            F.collect_set("source_anchor_business_id").alias("matched_anchor_ids")
        )
        .withColumn(
            "similarity_score",
            F.round(F.col("content_score"), 6)
        )
    )

    similar_limit = max(
        1,
        int(round(number_of_recommendations * SIMILAR_RESULT_RATIO))
    )
    exploration_limit = max(
        0,
        int(number_of_recommendations) - similar_limit
    )

    category_window = Window.partitionBy("categories").orderBy(
        F.desc("content_score"),
        F.desc("stars_clean"),
        F.desc("review_count_clean")
    )

    similar_results_df = (
        unseen_content_df
        .withColumn("same_category_rank", F.row_number().over(category_window))
        .filter(F.col("same_category_rank") <= 2)
        .orderBy(
            F.desc("content_score"),
            F.desc("stars_clean"),
            F.desc("review_count_clean")
        )
        .limit(similar_limit)
        .drop("same_category_rank")
        .withColumn("recommendation_type", F.lit("content_similar"))
    )

    selected_ids_df = similar_results_df.select(
        F.col("business_id").alias("selected_business_id")
    )

    exploration_pool_df = (
        candidate_df
        .join(
            already_seen_df,
            F.col("business_id") == F.col("seen_business_id"),
            "left_anti"
        )
        .join(
            selected_ids_df,
            F.col("business_id") == F.col("selected_business_id"),
            "left_anti"
        )
        .withColumn(
            "quality_score",
            (F.col("stars_clean") / F.lit(5.0)) +
            (F.log1p(F.col("review_count_clean")) / F.lit(20.0))
        )
        .orderBy(
            F.desc("quality_score"),
            F.desc("stars_clean"),
            F.desc("review_count_clean")
        )
        .limit(exploration_limit)
        .withColumn("content_score", F.lit(None).cast("double"))
        .withColumn("distance", F.lit(None).cast("double"))
        .withColumn("matched_anchor_ids", F.array().cast("array<string>"))
        .withColumn("similarity_score", F.lit(None).cast("double"))
        .withColumn("recommendation_type", F.lit("diversity_fallback"))
        .select(similar_results_df.columns)
    )

    return (
        similar_results_df
        .unionByName(exploration_pool_df)
        .withColumn("query_user_id", F.lit(user_id))
        .withColumn(
            "recommendation_reason",
            F.when(
                F.col("recommendation_type") == "content_similar",
                F.lit(
                    "Matches one or more businesses the user rated highly "
                    "and has not been rated by this user."
                )
            ).otherwise(
                F.lit(
                    "Diversity fallback: a highly rated, popular, open and "
                    "unseen business in the requested state."
                )
            )
        )
        .select(
            "query_user_id", "recommendation_type",
            "matched_anchor_ids", "business_id", "categories",
            "city", "state",
            F.col("latitude_clean").alias("latitude"),
            F.col("longitude_clean").alias("longitude"),
            F.col("stars_clean").alias("stars"),
            F.col("review_count_clean").alias("review_count"),
            "price_range_clean", "wifi_clean",
            "outdoor_seating_clean", "distance",
            "similarity_score", "recommendation_reason"
        )
        .limit(int(number_of_recommendations))
    )


## Step 14 — Optional cold-start recommendation function

This supports a new user who has no rating history but asks the RAG chatbot for a category and state.

In [ ]:
def recommend_for_cold_start(
    category_text,
    state,
    number_of_recommendations=10,
    price_range=None,
    wifi=None,
    outdoor_seating=None
):
    candidates = best_feature_df.filter(
        (F.col("is_open_clean") == "open") &
        (F.col("state_clean") == state.strip().upper()) &
        F.lower(F.col("categories")).contains(category_text.strip().lower())
    )

    if price_range is not None:
        candidates = candidates.filter(F.col("price_range_clean") == str(price_range).lower())
    if wifi is not None:
        candidates = candidates.filter(F.col("wifi_clean") == str(wifi).lower())
    if outdoor_seating is not None:
        candidates = candidates.filter(F.col("outdoor_seating_clean") == str(outdoor_seating).lower())

    return (
        candidates
        .withColumn("popularity_score", F.log1p(F.col("review_count_clean")))
        .orderBy(F.desc("stars_clean"), F.desc("popularity_score"))
        .select(
            "business_id", "categories", "city", "state",
            F.col("latitude_clean").alias("latitude"),
            F.col("longitude_clean").alias("longitude"),
            F.col("stars_clean").alias("stars"),
            F.col("review_count_clean").alias("review_count"),
            "price_range_clean", "wifi_clean", "outdoor_seating_clean"
        )
        .limit(int(number_of_recommendations))
    )

## Step 15 — Test the user recommendation function

A user with positive history is selected automatically for a safe notebook test.

In [ ]:
sample_user_row = (
    positive_interactions_df.groupBy("user_id")
    .count()
    .filter(F.col("count") >= 1)
    .orderBy(F.desc("count"), F.asc("user_id"))
    .first()
)

if sample_user_row is None:
    raise ValueError("No user with a positive interaction was found.")

TEST_USER_ID = sample_user_row["user_id"]
print("Test user:", TEST_USER_ID)

sample_user_recommendations_df = recommend_for_user(
    user_id=TEST_USER_ID,
    number_of_recommendations=TOP_K,
    requested_state=None,
    open_only=True
)

sample_user_recommendations_df.show(TOP_K, truncate=False)

## Methodological summary

- **Business catalogue:** used to fit the unsupervised content representation.
- **Validation users:** used only to compare hyperparameter combinations.
- **Test users:** used only once for final reported metrics.
- **Tuning method:** compact grid search.
- **Distance:** Euclidean distance over L2-normalized vectors.
- **Recommendation policy:** multi-anchor similarity + unseen filtering + controlled diversity fallback.


## Step 16 — Save final artifacts to S3

Saved artifacts:

- business feature vectors;
- fitted feature pipeline;
- selected LSH model;
- hyperparameter results;
- best parameters;
- final evaluation metrics;
- sample RAG-ready recommendations.

In [ ]:
try:
    (
        best_feature_df.write.mode("overwrite")
        .option("compression", "snappy")
        .parquet(FEATURES_OUTPUT_PATH)
    )

    best_pipeline_model.write().overwrite().save(PIPELINE_MODEL_PATH)
    best_lsh_model.write().overwrite().save(LSH_MODEL_PATH)

    tuning_results_df.write.mode("overwrite").parquet(TUNING_RESULTS_PATH)
    best_params_df.write.mode("overwrite").parquet(BEST_PARAMS_PATH)
    final_metrics_df.write.mode("overwrite").parquet(FINAL_METRICS_PATH)

    (
        sample_user_recommendations_df.write.mode("overwrite")
        .option("compression", "snappy")
        .parquet(RAG_RECOMMENDATIONS_PATH)
    )
except Exception:
    print("Failed while saving one or more final artifacts to S3.")
    print("Check s3:PutObject, s3:DeleteObject and model-output paths.")
    raise

print("All final artifacts saved successfully.")

## Step 17 — Verify saved outputs

In [ ]:
verified_features_df = spark.read.parquet(FEATURES_OUTPUT_PATH)
verified_metrics_df = spark.read.parquet(FINAL_METRICS_PATH)
verified_rag_df = spark.read.parquet(RAG_RECOMMENDATIONS_PATH)

print("Saved feature rows:", verified_features_df.count())
verified_metrics_df.show(truncate=False)
verified_rag_df.show(TOP_K, truncate=False)

print("Final content-based recommendation workflow completed successfully.")